# M2 학습예산·L2 진단 — 생략 없는 100·300 epoch 판독표

학습은 하지 않습니다. 완료된 L2 `1e-3`와 `1e-4` curve CSV를 읽어 100·300 epoch의 전체·CLV 구간별 정확도, 가격·구매금액 가중 지표, 노출지표와 작동 진단을 긴 형식으로 출력합니다. pandas의 `...` 생략 없이 원본값과 `M2−M1`을 함께 확인합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from pathlib import Path

PATHS = {
 'baseline_l2_1e-3': Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_clv_m2_capacity_search_v1/clv_m2_capacity_search_5f4b9c6e45d2_curve.csv'),
 'light_l2_1e-4': Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_clv_m2_capacity_search_light_l2_parallel_v1/clv_m2_capacity_search_5f4b9c6e45d2_curve.csv'),
}
for name,path in PATHS.items():
    assert path.exists(), f'{name} 결과가 없습니다: {path}'
frames=[]
for name,path in PATHS.items():
    part=pd.read_csv(path); part.insert(0,'run',name); frames.append(part)
all_curve=pd.concat(frames,ignore_index=True)
assert set(all_curve.epoch.astype(int)) >= {100,300}
print('읽은 행/열:', all_curve.shape)

In [ ]:
# 논문 판독 대상 열만 선택하되 값은 생략하지 않는다.
base_metrics = [
 'recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
 'price_purchase_amount_weighted_hit@10','vndcg@10',
 'price_purchase_amount_weighted_hit@20','vndcg@20',
 'price_purchase_amount_weighted_hit@50','vndcg@50',
 'coverage@10','coverage@20','coverage@50',
 'user_value_tendency_recommended_price_alignment',
]
segment_suffixes = (
 'recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
 'revenue@10','vndcg@10','revenue@20','vndcg@20','revenue@50','vndcg@50'
)
segment_metrics = [c for c in all_curve.columns
                   if c.startswith(('저CLV_','중CLV_','고CLV_')) and c.endswith(segment_suffixes)]
diagnostics = [c for c in (
 'loss','p_correct','id_score_mean_abs','clv_score_mean_abs','clv_score_share',
 'activity_source_gradient_norm','activity_target_gradient_norm',
 'value_source_gradient_norm','value_target_gradient_norm') if c in all_curve]
metrics=[c for c in base_metrics+segment_metrics+diagnostics if c in all_curve]
missing=[c for c in base_metrics if c not in all_curve]
assert not missing, f'필수 전체지표 누락: {missing}'
selected=all_curve[all_curve.epoch.isin([100,300])][['run','model_id','epoch',*metrics]].copy()
print('1) 100·300 epoch 절대지표 — transpose, 생략 없음')
print(selected.set_index(['run','model_id','epoch']).T.to_string())

In [ ]:
rows=[]
for (run,epoch),part in selected.groupby(['run','epoch']):
    m1=part[part.model_id.eq('m1_bpr_k1')].iloc[0]
    m2=part[part.model_id.eq('m2_nv_history_fit_bpr_k1')].iloc[0]
    for metric in metrics:
        if metric in diagnostics: continue
        base=float(m1[metric]); value=float(m2[metric])
        rows.append({'run':run,'epoch':int(epoch),'metric':metric,'M1':base,'M2':value,
                     'absolute_delta':value-base,
                     'relative_change_pct':100*(value-base)/base if base else float('nan')})
comparison=pd.DataFrame(rows)
print('2) M2−M1 전체 비교 — 생략 없음')
print(comparison.to_string(index=False))

out=Path('/content/drive/MyDrive/논문/data/m2_learning_budget_complete_readout.csv')
comparison.to_csv(out,index=False)
print('저장:',out)